# Arabic Text Classification with Transformer Fine-Tuning

This notebook provides a **production-ready, reusable pipeline** for **Arabic text classification** using **Hugging Face Transformers** and **PyTorch**.

It is designed to run **top-to-bottom** in Jupyter Notebook and covers:

- Problem understanding
- Data ingestion
- EDA
- Arabic-aware preprocessing
- Train/validation/test split
- Class imbalance handling
- Tokenization
- Dataset preparation
- Fine-tuning
- Hyperparameter tuning
- Evaluation
- Saving and inference

---
## Overview

This notebook is optimized for **Arabic-language text classification**. It supports:

- **Binary classification**
- **Multi-class classification**
- **Multi-label classification** (basic pathway included)

It focuses on practical implementation, reproducibility, and clean organization.

## Assumptions

- Input data contains at least two columns:
  - `text`
  - `label`
- Text is primarily **Arabic**.
- Data may come from **CSV**, **Excel**, or an existing **pandas DataFrame**.
- The default full training path is optimized for **single-label classification**.
- Multi-label support is included, but production multi-label projects may require:
  - threshold tuning
  - iterative stratification
  - additional metric monitoring
- GPU is preferred but not required.

## Notebook Structure

1. Imports  
2. Configuration  
3. Problem Understanding  
4. Data Loading  
5. EDA  
6. Preprocessing  
7. Train / Validation / Test Split  
8. Class Imbalance Handling  
9. Tokenization  
10. Dataset Preparation  
11. Model Setup  
12. Training  
13. Hyperparameter Tuning  
14. Evaluation  
15. Save Model  
16. Inference Examples  
17. Best Practices, Pitfalls, and Future Improvements

## 1. Imports

In [ ]:
# Core
import os
import re
import json
import random
import warnings
from pathlib import Path
from collections import Counter

# Data
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt

# Sklearn
from sklearn.model_selection import train_test_split, ParameterGrid
from sklearn.preprocessing import LabelEncoder, MultiLabelBinarizer
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
    roc_auc_score
)
from sklearn.utils.class_weight import compute_class_weight

# Optional imbalance helpers
from imblearn.over_sampling import RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler

# Hugging Face / PyTorch
import torch
import torch.nn as nn
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
    set_seed
)

warnings.filterwarnings("ignore")

## 2. Configuration

In [ ]:
class CFG:
    # =========================
    # Data
    # =========================
    data_path = "your_dataset.csv"       # Change this
    file_type = "csv"                    # Options: csv, excel, dataframe
    text_col = "text"
    label_col = "label"

    # =========================
    # Task settings
    # =========================
    task_type = "auto"                   # auto | binary | multiclass | multilabel
    multilabel_delimiter = ","           # only used if label column contains strings like "sports,politics"

    # =========================
    # Arabic preprocessing
    # =========================
    remove_urls = True
    normalize_whitespace = True
    remove_emojis = False                # usually keep unless noisy
    remove_tatweel = True
    normalize_arabic = True
    normalize_alef = True
    normalize_yeh = True
    normalize_teh_marbuta = False        # set carefully; may remove useful lexical signal
    keep_punctuation = True

    # =========================
    # Split
    # =========================
    train_size = 0.70
    val_size = 0.15
    test_size = 0.15
    random_state = 42

    # =========================
    # Model candidates (Arabic)
    # =========================
    model_candidates = {
        "distil_arabic": "asafaya/bert-mini-arabic",
        "arabert_v2": "aubmindlab/bert-base-arabertv02",
        "marbert_v2": "UBC-NLP/MARBERTv2",
        "camelbert_msa": "CAMeL-Lab/bert-base-arabic-camelbert-mix",
        "arabicbert": "asafaya/bert-base-arabic",
    }
    baseline_model_key = "arabert_v2"

    # =========================
    # Tokenization
    # =========================
    max_length = None                    # auto-compute if None
    max_length_quantile = 0.95
    hard_max_length_cap = 256

    # =========================
    # Training
    # =========================
    output_dir = "outputs_arabic_cls"
    num_train_epochs = 4
    learning_rate = 2e-5
    weight_decay = 0.01
    train_batch_size = 16
    eval_batch_size = 32
    gradient_accumulation_steps = 1
    warmup_ratio = 0.1
    logging_steps = 50
    save_total_limit = 2
    load_best_model_at_end = True
    metric_for_best_model = "eval_f1_weighted"
    greater_is_better = True
    fp16 = torch.cuda.is_available()

    # =========================
    # Imbalance handling
    # =========================
    imbalance_ratio_threshold = 1.5
    minority_fraction_threshold = 0.20
    imbalance_strategy = "weighted_loss"   # weighted_loss | oversample | undersample | none

    # =========================
    # Hyperparameter search
    # =========================
    do_hparam_search = True
    hparam_grid = {
        "learning_rate": [2e-5, 3e-5],
        "num_train_epochs": [3, 4],
        "weight_decay": [0.0, 0.01],
        "train_batch_size": [16],
    }

    # =========================
    # Saving
    # =========================
    save_dir = "final_arabic_text_classifier"

    # =========================
    # Reproducibility
    # =========================
    seed = 42

set_seed(CFG.seed)
random.seed(CFG.seed)
np.random.seed(CFG.seed)
torch.manual_seed(CFG.seed)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

## 3. Problem Understanding

A text classification task can be identified as:

- **Binary classification**: exactly 2 unique target classes
- **Multi-class classification**: more than 2 unique target classes, with one label per sample
- **Multi-label classification**: each sample can have **multiple labels** simultaneously

### How to determine this from the dataset

- Inspect the `label` column:
  - if each row has a single label and there are 2 unique classes → binary
  - if each row has a single label and there are >2 unique classes → multi-class
  - if rows contain multiple labels (lists, sets, or delimited strings) → multi-label

### Inputs and targets

- **Input**: Arabic text from the `text` column
- **Target**: class label(s) from the `label` column

In [ ]:
def load_data(path=None, file_type="csv", dataframe=None):
    if file_type == "csv":
        df = pd.read_csv(path)
    elif file_type == "excel":
        df = pd.read_excel(path)
    elif file_type == "dataframe":
        if dataframe is None:
            raise ValueError("For file_type='dataframe', provide a pandas DataFrame.")
        df = dataframe.copy()
    else:
        raise ValueError("Unsupported file_type. Use 'csv', 'excel', or 'dataframe'.")
    return df


def infer_task_type(df, label_col, user_task_type="auto", multilabel_delimiter=","):
    if user_task_type != "auto":
        return user_task_type

    sample = df[label_col].dropna().iloc[0]

    # Already list-like
    if isinstance(sample, (list, tuple, set)):
        return "multilabel"

    # Delimited multilabel strings
    if isinstance(sample, str) and multilabel_delimiter in sample:
        multi_count = df[label_col].dropna().astype(str).str.contains(re.escape(multilabel_delimiter)).mean()
        if multi_count > 0.3:
            return "multilabel"

    n_unique = df[label_col].nunique(dropna=True)
    if n_unique == 2:
        return "binary"
    elif n_unique > 2:
        return "multiclass"
    else:
        raise ValueError("Could not infer task type. Check the label column.")

## 4. Data Loading

In [ ]:
df = load_data(path=CFG.data_path, file_type=CFG.file_type)

required_cols = {CFG.text_col, CFG.label_col}
missing_required = required_cols - set(df.columns)
if missing_required:
    raise ValueError(f"Dataset is missing required columns: {missing_required}")

task_type = infer_task_type(df, CFG.label_col, CFG.task_type, CFG.multilabel_delimiter)

print("Detected task type:", task_type)
print("\nHead:")
display(df.head())

print("Shape:", df.shape)
print("\nDtypes:")
display(df.dtypes)

print("\nMissing values:")
display(df.isnull().sum())

## 5. EDA

In [ ]:
eda_df = df.copy()

# Basic missingness
print("Missing values:")
display(eda_df[[CFG.text_col, CFG.label_col]].isnull().sum().to_frame("missing_count"))

# Duplicate rows
duplicate_count = eda_df.duplicated(subset=[CFG.text_col, CFG.label_col]).sum()
print("Duplicate rows (text + label):", duplicate_count)

# Text length stats
eda_df["char_len"] = eda_df[CFG.text_col].astype(str).apply(len)
eda_df["word_len"] = eda_df[CFG.text_col].astype(str).apply(lambda x: len(str(x).split()))

print("\nText length summary (characters):")
display(eda_df["char_len"].describe())

print("\nText length summary (words):")
display(eda_df["word_len"].describe())

In [ ]:
# Plot text length distributions
plt.figure(figsize=(10, 4))
plt.hist(eda_df["char_len"], bins=40)
plt.title("Arabic Text Length Distribution (Characters)")
plt.xlabel("Character count")
plt.ylabel("Frequency")
plt.show()

plt.figure(figsize=(10, 4))
plt.hist(eda_df["word_len"], bins=40)
plt.title("Arabic Text Length Distribution (Words)")
plt.xlabel("Word count")
plt.ylabel("Frequency")
plt.show()

In [ ]:
if task_type in ["binary", "multiclass"]:
    class_counts = eda_df[CFG.label_col].value_counts(dropna=False)
    print("Class distribution:")
    display(class_counts.to_frame("count"))

    plt.figure(figsize=(10, 4))
    class_counts.plot(kind="bar")
    plt.title("Class Distribution")
    plt.xlabel("Label")
    plt.ylabel("Count")
    plt.show()

    max_count = class_counts.max()
    min_count = class_counts.min()
    imbalance_ratio = max_count / max(min_count, 1)
    minority_fraction = min_count / class_counts.sum()

    is_imbalanced = (
        imbalance_ratio >= CFG.imbalance_ratio_threshold
        or minority_fraction <= CFG.minority_fraction_threshold
    )

    print(f"Imbalance ratio (max/min): {imbalance_ratio:.2f}")
    print(f"Minority fraction: {minority_fraction:.4f}")
    print("Imbalanced dataset:", is_imbalanced)

else:
    print("Multi-label class distribution will be analyzed after binarization.")

### Arabic preprocessing notes

For Transformer models, **light preprocessing** is usually best.

Useful Arabic-specific normalization may include:

- removing URLs
- collapsing whitespace
- removing tatweel `ـ`
- optionally normalizing variant Arabic letters:
  - Alef variants: `أ`, `إ`, `آ` → `ا`
  - Yeh variants: `ى` → `ي`
  - Teh Marbuta: `ة` → `ه` (**use carefully**)

### When Arabic normalization helps

Normalization often helps when:

- the dataset mixes multiple spelling forms
- user-generated content is noisy
- retrieval of lexical consistency matters

### When normalization may remove useful signal

Over-normalization may hurt performance when:

- morphology matters
- named entities matter
- dialect or domain-specific spellings matter
- `ة` vs `ه` distinctions carry useful meaning

### Important Arabic note

**Arabic dialectal variation matters.** If the data is dialect-heavy, models such as **MARBERT** may outperform models more optimized for MSA. If the data is formal / news / governmental / academic Arabic, **AraBERT** or **CAMeLBERT** often make strong baselines.

## 6. Preprocessing

In [ ]:
URL_PATTERN = re.compile(r"https?://\S+|www\.\S+")
WHITESPACE_PATTERN = re.compile(r"\s+")
TATWEEL_PATTERN = re.compile(r"ـ+")
EMOJI_PATTERN = re.compile(
    "["
    "\U0001F600-\U0001F64F"
    "\U0001F300-\U0001F5FF"
    "\U0001F680-\U0001F6FF"
    "\U0001F1E0-\U0001F1FF"
    "\U00002700-\U000027BF"
    "\U000024C2-\U0001F251"
    "]+",
    flags=re.UNICODE
)

ALEF_VARIANTS_PATTERN = re.compile(r"[إأآٱ]")
YEH_VARIANTS_PATTERN = re.compile(r"ى")
TEH_MARBUTA_PATTERN = re.compile(r"ة")

def normalize_arabic_text(text,
                          remove_urls=True,
                          normalize_whitespace=True,
                          remove_emojis=False,
                          remove_tatweel=True,
                          normalize_arabic=True,
                          normalize_alef=True,
                          normalize_yeh=True,
                          normalize_teh_marbuta=False):
    text = str(text)

    if remove_urls:
        text = URL_PATTERN.sub(" ", text)

    if remove_emojis:
        text = EMOJI_PATTERN.sub(" ", text)

    if remove_tatweel:
        text = TATWEEL_PATTERN.sub("", text)

    if normalize_arabic:
        if normalize_alef:
            text = ALEF_VARIANTS_PATTERN.sub("ا", text)
        if normalize_yeh:
            text = YEH_VARIANTS_PATTERN.sub("ي", text)
        if normalize_teh_marbuta:
            text = TEH_MARBUTA_PATTERN.sub("ه", text)

    if normalize_whitespace:
        text = WHITESPACE_PATTERN.sub(" ", text).strip()

    return text

In [ ]:
# Keep raw copy
df["text_raw"] = df[CFG.text_col].astype(str)

# Apply light preprocessing
df[CFG.text_col] = df[CFG.text_col].astype(str).apply(
    lambda x: normalize_arabic_text(
        x,
        remove_urls=CFG.remove_urls,
        normalize_whitespace=CFG.normalize_whitespace,
        remove_emojis=CFG.remove_emojis,
        remove_tatweel=CFG.remove_tatweel,
        normalize_arabic=CFG.normalize_arabic,
        normalize_alef=CFG.normalize_alef,
        normalize_yeh=CFG.normalize_yeh,
        normalize_teh_marbuta=CFG.normalize_teh_marbuta,
    )
)

# Drop rows missing essential fields after cleaning
df = df.dropna(subset=[CFG.text_col, CFG.label_col]).copy()
df = df[df[CFG.text_col].str.strip().astype(bool)].copy()

print("Preview after preprocessing:")
display(df[[CFG.text_col, "text_raw", CFG.label_col]].head())

## 7. Train / Validation / Test Split

In [ ]:
def split_single_label_data(df, text_col, label_col, random_state=42):
    train_df, temp_df = train_test_split(
        df,
        test_size=(1 - CFG.train_size),
        stratify=df[label_col],
        random_state=random_state
    )

    relative_val_size = CFG.val_size / (CFG.val_size + CFG.test_size)
    val_df, test_df = train_test_split(
        temp_df,
        test_size=(1 - relative_val_size),
        stratify=temp_df[label_col],
        random_state=random_state
    )
    return train_df.reset_index(drop=True), val_df.reset_index(drop=True), test_df.reset_index(drop=True)


def parse_multilabel_column(series, delimiter=","):
    parsed = []
    for item in series.astype(str).tolist():
        labels = [x.strip() for x in item.split(delimiter) if x.strip()]
        parsed.append(labels)
    return parsed

In [ ]:
if task_type in ["binary", "multiclass"]:
    train_df, val_df, test_df = split_single_label_data(
        df, CFG.text_col, CFG.label_col, CFG.random_state
    )
else:
    # Simple random split for multi-label.
    # For production multi-label work, iterative stratification is recommended.
    train_df, temp_df = train_test_split(
        df,
        test_size=(1 - CFG.train_size),
        random_state=CFG.random_state
    )
    relative_val_size = CFG.val_size / (CFG.val_size + CFG.test_size)
    val_df, test_df = train_test_split(
        temp_df,
        test_size=(1 - relative_val_size),
        random_state=CFG.random_state
    )

    train_df = train_df.reset_index(drop=True)
    val_df = val_df.reset_index(drop=True)
    test_df = test_df.reset_index(drop=True)

print("Train shape:", train_df.shape)
print("Validation shape:", val_df.shape)
print("Test shape:", test_df.shape)

### Why this split?

Default split ratio:

- **70% train**
- **15% validation**
- **15% test**

Why:

- enough data for training
- a dedicated validation set for checkpoint/model selection
- a final untouched test set for honest evaluation

For single-label classification, **stratification** preserves class proportions and improves evaluation reliability on imbalanced datasets.

## 8. Class Imbalance Handling

In [ ]:
if task_type in ["binary", "multiclass"]:
    train_class_counts = train_df[CFG.label_col].value_counts()
    max_count = train_class_counts.max()
    min_count = train_class_counts.min()
    train_imbalance_ratio = max_count / max(min_count, 1)
    train_minority_fraction = min_count / train_class_counts.sum()

    train_is_imbalanced = (
        train_imbalance_ratio >= CFG.imbalance_ratio_threshold
        or train_minority_fraction <= CFG.minority_fraction_threshold
    )

    print("Training class counts:")
    display(train_class_counts.to_frame("count"))

    print(f"Training imbalance ratio (max/min): {train_imbalance_ratio:.2f}")
    print(f"Training minority fraction: {train_minority_fraction:.4f}")
    print("Imbalanced training data:", train_is_imbalanced)
else:
    train_is_imbalanced = False
    print("For multi-label problems, imbalance should be measured per label after binarization.")

### Comparing imbalance strategies

Possible options:

1. **Weighted loss / class weights**
   - Best default when using Transformers
   - Keeps original data distribution
   - Avoids repeating noisy minority examples too much

2. **Oversampling**
   - Can help rare classes
   - Risk: duplicated minority examples may overfit

3. **Undersampling**
   - Simple
   - May discard useful Arabic text data

4. **Data augmentation**
   - Possible, but should be used carefully for Arabic
   - Synonym replacement and back-translation can introduce unnatural phrasing

### Recommended default

For Transformer fine-tuning, start with **weighted cross-entropy**. It is usually the cleanest and most stable first choice.

In [ ]:
# Encode labels
if task_type in ["binary", "multiclass"]:
    label_encoder = LabelEncoder()
    train_df["label_id"] = label_encoder.fit_transform(train_df[CFG.label_col])
    val_df["label_id"] = label_encoder.transform(val_df[CFG.label_col])
    test_df["label_id"] = label_encoder.transform(test_df[CFG.label_col])

    id2label = {i: label for i, label in enumerate(label_encoder.classes_)}
    label2id = {label: i for i, label in id2label.items()}

    class_weights = compute_class_weight(
        class_weight="balanced",
        classes=np.unique(train_df["label_id"]),
        y=train_df["label_id"]
    )
    class_weights_tensor = torch.tensor(class_weights, dtype=torch.float)
    print("Label mapping:", label2id)
    print("Class weights:", class_weights)

else:
    train_labels = parse_multilabel_column(train_df[CFG.label_col], CFG.multilabel_delimiter)
    val_labels = parse_multilabel_column(val_df[CFG.label_col], CFG.multilabel_delimiter)
    test_labels = parse_multilabel_column(test_df[CFG.label_col], CFG.multilabel_delimiter)

    mlb = MultiLabelBinarizer()
    train_y = mlb.fit_transform(train_labels)
    val_y = mlb.transform(val_labels)
    test_y = mlb.transform(test_labels)

    train_df["labels"] = train_y.tolist()
    val_df["labels"] = val_y.tolist()
    test_df["labels"] = test_y.tolist()

    id2label = {i: label for i, label in enumerate(mlb.classes_)}
    label2id = {label: i for i, label in id2label.items()}

    label_freq = train_y.sum(axis=0)
    pos_weight = (len(train_y) - label_freq) / np.maximum(label_freq, 1)
    class_weights_tensor = torch.tensor(pos_weight, dtype=torch.float)

    print("Multi-label classes:", list(mlb.classes_))

In [ ]:
# Optional comparison: oversampling for single-label tasks
if task_type in ["binary", "multiclass"] and CFG.imbalance_strategy == "oversample":
    ros = RandomOverSampler(random_state=CFG.random_state)
    X_resampled, y_resampled = ros.fit_resample(
        train_df[[CFG.text_col]],
        train_df["label_id"]
    )
    train_df = pd.DataFrame({
        CFG.text_col: X_resampled[CFG.text_col],
        "label_id": y_resampled
    })
    train_df[CFG.label_col] = train_df["label_id"].map(id2label)
    print("Applied RandomOverSampler.")
    display(train_df["label_id"].value_counts().sort_index().to_frame("count"))

elif task_type in ["binary", "multiclass"] and CFG.imbalance_strategy == "undersample":
    rus = RandomUnderSampler(random_state=CFG.random_state)
    X_resampled, y_resampled = rus.fit_resample(
        train_df[[CFG.text_col]],
        train_df["label_id"]
    )
    train_df = pd.DataFrame({
        CFG.text_col: X_resampled[CFG.text_col],
        "label_id": y_resampled
    })
    train_df[CFG.label_col] = train_df["label_id"].map(id2label)
    print("Applied RandomUnderSampler.")
    display(train_df["label_id"].value_counts().sort_index().to_frame("count"))

## 9. Tokenization

### Arabic model selection guidance

Common Arabic model choices:

- **AraBERT**  
  Strong baseline for Modern Standard Arabic and many general Arabic tasks.

- **MARBERT**  
  Strong choice for dialect-heavy Arabic and social media text.

- **CAMeLBERT**  
  Good family of Arabic models from CAMeL Lab; often useful depending on domain.

- **ArabicBERT / smaller Arabic BERTs**  
  Useful when deployment constraints matter.

### Practical baseline recommendation

- Start with **AraBERT** for formal or mixed Arabic
- Try **MARBERT** if the dataset is highly dialectal, user-generated, or social-media-heavy

In [ ]:
selected_model_name = CFG.model_candidates[CFG.baseline_model_key]
print("Selected baseline model:", selected_model_name)

tokenizer = AutoTokenizer.from_pretrained(selected_model_name, use_fast=True)

In [ ]:
def estimate_max_length(texts, tokenizer, quantile=0.95, cap=256):
    lengths = []
    sample_texts = list(texts)
    for text in sample_texts:
        token_ids = tokenizer(
            str(text),
            truncation=False,
            padding=False,
            add_special_tokens=True
        )["input_ids"]
        lengths.append(len(token_ids))
    est = int(np.quantile(lengths, quantile))
    return min(max(est, 32), cap), lengths

if CFG.max_length is None:
    estimated_max_length, token_lengths = estimate_max_length(
        train_df[CFG.text_col].tolist(),
        tokenizer,
        quantile=CFG.max_length_quantile,
        cap=CFG.hard_max_length_cap
    )
    CFG.max_length = estimated_max_length

print("Chosen max_length:", CFG.max_length)

plt.figure(figsize=(10, 4))
plt.hist(token_lengths, bins=40)
plt.title("Token Length Distribution")
plt.xlabel("Number of tokens")
plt.ylabel("Frequency")
plt.show()

## 10. Dataset Preparation

In [ ]:
def convert_to_hf_dataset(df, text_col, label_col_name):
    keep_cols = [text_col, label_col_name]
    return Dataset.from_pandas(df[keep_cols], preserve_index=False)

if task_type in ["binary", "multiclass"]:
    train_hf = convert_to_hf_dataset(train_df, CFG.text_col, "label_id")
    val_hf = convert_to_hf_dataset(val_df, CFG.text_col, "label_id")
    test_hf = convert_to_hf_dataset(test_df, CFG.text_col, "label_id")
else:
    train_hf = convert_to_hf_dataset(train_df, CFG.text_col, "labels")
    val_hf = convert_to_hf_dataset(val_df, CFG.text_col, "labels")
    test_hf = convert_to_hf_dataset(test_df, CFG.text_col, "labels")

dataset_dict = DatasetDict({
    "train": train_hf,
    "validation": val_hf,
    "test": test_hf
})

dataset_dict

In [ ]:
label_field = "label_id" if task_type in ["binary", "multiclass"] else "labels"

def tokenize_batch(batch):
    return tokenizer(
        batch[CFG.text_col],
        truncation=True,
        max_length=CFG.max_length
    )

tokenized_ds = dataset_dict.map(tokenize_batch, batched=True)

if task_type in ["binary", "multiclass"]:
    tokenized_ds = tokenized_ds.rename_column("label_id", "labels")

tokenized_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

tokenized_ds

## 11. Model Setup

In [ ]:
num_labels = len(label2id)
problem_type = (
    "multi_label_classification" if task_type == "multilabel"
    else "single_label_classification"
)

model = AutoModelForSequenceClassification.from_pretrained(
    selected_model_name,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
    problem_type=problem_type
)

model.config.id2label = id2label
model.config.label2id = label2id

print(model.config)

In [ ]:
def compute_metrics_single_label(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    acc = accuracy_score(labels, preds)
    precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(
        labels, preds, average="macro", zero_division=0
    )
    precision_weighted, recall_weighted, f1_weighted, _ = precision_recall_fscore_support(
        labels, preds, average="weighted", zero_division=0
    )

    metrics = {
        "accuracy": acc,
        "precision_macro": precision_macro,
        "recall_macro": recall_macro,
        "f1_macro": f1_macro,
        "precision_weighted": precision_weighted,
        "recall_weighted": recall_weighted,
        "f1_weighted": f1_weighted,
    }

    # ROC-AUC if binary
    if len(np.unique(labels)) == 2:
        probs = torch.softmax(torch.tensor(logits), dim=1).numpy()[:, 1]
        try:
            metrics["roc_auc"] = roc_auc_score(labels, probs)
        except Exception:
            pass

    return metrics


def compute_metrics_multilabel(eval_pred, threshold=0.5):
    logits, labels = eval_pred
    probs = 1 / (1 + np.exp(-logits))
    preds = (probs >= threshold).astype(int)

    precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(
        labels, preds, average="macro", zero_division=0
    )
    precision_weighted, recall_weighted, f1_weighted, _ = precision_recall_fscore_support(
        labels, preds, average="weighted", zero_division=0
    )

    subset_acc = (preds == labels).all(axis=1).mean()

    metrics = {
        "subset_accuracy": subset_acc,
        "precision_macro": precision_macro,
        "recall_macro": recall_macro,
        "f1_macro": f1_macro,
        "precision_weighted": precision_weighted,
        "recall_weighted": recall_weighted,
        "f1_weighted": f1_weighted,
    }

    try:
        metrics["roc_auc_macro"] = roc_auc_score(labels, probs, average="macro")
    except Exception:
        pass

    return metrics


compute_metrics_fn = compute_metrics_multilabel if task_type == "multilabel" else compute_metrics_single_label

In [ ]:
class WeightedLossTrainer(Trainer):
    def __init__(self, class_weights=None, task_type="single_label_classification", *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights
        self.task_type = task_type

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"]
        )
        logits = outputs.get("logits")

        if self.task_type == "multi_label_classification":
            if self.class_weights is not None:
                loss_fct = nn.BCEWithLogitsLoss(pos_weight=self.class_weights.to(logits.device))
            else:
                loss_fct = nn.BCEWithLogitsLoss()
            loss = loss_fct(logits, labels.float())
        else:
            if self.class_weights is not None:
                loss_fct = nn.CrossEntropyLoss(weight=self.class_weights.to(logits.device))
            else:
                loss_fct = nn.CrossEntropyLoss()
            loss = loss_fct(logits.view(-1, model.config.num_labels), labels.view(-1))

        return (loss, outputs) if return_outputs else loss

## 12. Training

In [ ]:
def build_training_args(output_dir,
                        learning_rate,
                        num_train_epochs,
                        weight_decay,
                        train_batch_size,
                        eval_batch_size):
    return TrainingArguments(
        output_dir=output_dir,
        eval_strategy="epoch",
        save_strategy="epoch",
        logging_strategy="steps",
        logging_steps=CFG.logging_steps,
        learning_rate=learning_rate,
        per_device_train_batch_size=train_batch_size,
        per_device_eval_batch_size=eval_batch_size,
        gradient_accumulation_steps=CFG.gradient_accumulation_steps,
        num_train_epochs=num_train_epochs,
        weight_decay=weight_decay,
        warmup_ratio=CFG.warmup_ratio,
        load_best_model_at_end=CFG.load_best_model_at_end,
        metric_for_best_model=CFG.metric_for_best_model,
        greater_is_better=CFG.greater_is_better,
        fp16=CFG.fp16,
        save_total_limit=CFG.save_total_limit,
        report_to="none",
        seed=CFG.seed
    )

In [ ]:
base_training_args = build_training_args(
    output_dir=os.path.join(CFG.output_dir, "baseline"),
    learning_rate=CFG.learning_rate,
    num_train_epochs=CFG.num_train_epochs,
    weight_decay=CFG.weight_decay,
    train_batch_size=CFG.train_batch_size,
    eval_batch_size=CFG.eval_batch_size
)

trainer = WeightedLossTrainer(
    model=model,
    args=base_training_args,
    train_dataset=tokenized_ds["train"],
    eval_dataset=tokenized_ds["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics_fn,
    class_weights=class_weights_tensor if CFG.imbalance_strategy == "weighted_loss" else None,
    task_type=problem_type,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

train_result = trainer.train()
trainer.save_model(os.path.join(CFG.output_dir, "baseline", "best_model"))

In [ ]:
print("Best checkpoint:", trainer.state.best_model_checkpoint)

val_metrics = trainer.evaluate(tokenized_ds["validation"])
print("Validation metrics:")
val_metrics

### What this training pipeline includes

- full Transformer fine-tuning
- learning-rate scheduling through `TrainingArguments`
- early stopping
- checkpointing
- metric-based best-model loading
- mixed precision when CUDA is available
- weighted loss for imbalance handling

## 13. Hyperparameter Tuning

In [ ]:
def train_one_run(params, model_name, run_name):
    local_model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=len(label2id),
        id2label=id2label,
        label2id=label2id,
        problem_type=problem_type
    )

    args = build_training_args(
        output_dir=os.path.join(CFG.output_dir, run_name),
        learning_rate=params["learning_rate"],
        num_train_epochs=params["num_train_epochs"],
        weight_decay=params["weight_decay"],
        train_batch_size=params["train_batch_size"],
        eval_batch_size=CFG.eval_batch_size
    )

    local_trainer = WeightedLossTrainer(
        model=local_model,
        args=args,
        train_dataset=tokenized_ds["train"],
        eval_dataset=tokenized_ds["validation"],
        tokenizer=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics_fn,
        class_weights=class_weights_tensor if CFG.imbalance_strategy == "weighted_loss" else None,
        task_type=problem_type,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
    )

    local_trainer.train()
    metrics = local_trainer.evaluate(tokenized_ds["validation"])
    return local_trainer, metrics

In [ ]:
search_results = []

if CFG.do_hparam_search:
    for i, params in enumerate(ParameterGrid(CFG.hparam_grid), start=1):
        print(f"Running search trial {i} / {len(list(ParameterGrid(CFG.hparam_grid)))}")
        print("Params:", params)

        run_name = f"search_run_{i}"
        local_trainer, metrics = train_one_run(params, selected_model_name, run_name)

        result = {"run_name": run_name, **params, **metrics}
        search_results.append(result)

    search_results_df = pd.DataFrame(search_results).sort_values(
        by=CFG.metric_for_best_model, ascending=False
    ).reset_index(drop=True)

    print("Hyperparameter search results:")
    display(search_results_df)

    best_params = search_results_df.iloc[0].to_dict()
else:
    search_results_df = pd.DataFrame()
    best_params = {
        "learning_rate": CFG.learning_rate,
        "num_train_epochs": CFG.num_train_epochs,
        "weight_decay": CFG.weight_decay,
        "train_batch_size": CFG.train_batch_size
    }

print("Best params:")
print(best_params)

### Practical tuning recommendation

Keep tuning **small and targeted** first:

- learning rate: usually most important
- batch size: affects stability and speed
- epochs: avoid overtraining
- weight decay: helps generalization

For Arabic tasks with limited GPU budget, a **small grid search** is often enough to get most of the gains.

## 14. Final Model Training and Evaluation

In [ ]:
final_model = AutoModelForSequenceClassification.from_pretrained(
    selected_model_name,
    num_labels=len(label2id),
    id2label=id2label,
    label2id=label2id,
    problem_type=problem_type
)

final_args = build_training_args(
    output_dir=os.path.join(CFG.output_dir, "final_model"),
    learning_rate=float(best_params["learning_rate"]),
    num_train_epochs=int(best_params["num_train_epochs"]),
    weight_decay=float(best_params["weight_decay"]),
    train_batch_size=int(best_params["train_batch_size"]),
    eval_batch_size=CFG.eval_batch_size
)

final_trainer = WeightedLossTrainer(
    model=final_model,
    args=final_args,
    train_dataset=tokenized_ds["train"],
    eval_dataset=tokenized_ds["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics_fn,
    class_weights=class_weights_tensor if CFG.imbalance_strategy == "weighted_loss" else None,
    task_type=problem_type,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

final_trainer.train()

In [ ]:
val_metrics = final_trainer.evaluate(tokenized_ds["validation"])
test_metrics = final_trainer.evaluate(tokenized_ds["test"])

print("Validation metrics:")
display(pd.DataFrame([val_metrics]))

print("Test metrics:")
display(pd.DataFrame([test_metrics]))

In [ ]:
# Detailed evaluation for single-label tasks
if task_type in ["binary", "multiclass"]:
    pred_output = final_trainer.predict(tokenized_ds["test"])
    y_true = pred_output.label_ids
    y_pred = np.argmax(pred_output.predictions, axis=1)
    y_prob = torch.softmax(torch.tensor(pred_output.predictions), dim=1).numpy()

    print("Classification report:")
    print(classification_report(
        y_true,
        y_pred,
        target_names=[id2label[i] for i in range(len(id2label))],
        zero_division=0
    ))

    cm = confusion_matrix(y_true, y_pred)
    print("Confusion matrix:")
    print(cm)

    plt.figure(figsize=(6, 5))
    plt.imshow(cm)
    plt.title("Confusion Matrix")
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.colorbar()
    plt.show()

    if len(np.unique(y_true)) == 2:
        try:
            binary_auc = roc_auc_score(y_true, y_prob[:, 1])
            print("Test ROC-AUC:", round(binary_auc, 4))
        except Exception as e:
            print("ROC-AUC could not be computed:", e)
else:
    pred_output = final_trainer.predict(tokenized_ds["test"])
    y_true = pred_output.label_ids
    y_prob = 1 / (1 + np.exp(-pred_output.predictions))
    y_pred = (y_prob >= 0.5).astype(int)

    print("Multi-label evaluation uses macro/weighted precision, recall, and F1.")
    print("Consider threshold tuning per label for production.")

### How to select the final model

Do **not** select the best model using accuracy alone, especially on imbalanced Arabic datasets.

Preferred metrics:

- **Weighted F1**: good default for imbalanced single-label tasks
- **Macro F1**: useful when minority classes matter a lot
- **ROC-AUC**: useful for binary tasks, but not a replacement for F1

## 15. Saving the Model, Tokenizer, and Label Mapping

In [ ]:
save_dir = Path(CFG.save_dir)
save_dir.mkdir(parents=True, exist_ok=True)

final_trainer.save_model(str(save_dir))
tokenizer.save_pretrained(str(save_dir))

with open(save_dir / "label_mapping.json", "w", encoding="utf-8") as f:
    json.dump(
        {
            "id2label": {str(k): v for k, v in id2label.items()},
            "label2id": label2id,
            "task_type": task_type,
            "max_length": CFG.max_length,
            "model_name": selected_model_name
        },
        f,
        ensure_ascii=False,
        indent=2
    )

print(f"Saved artifacts to: {save_dir.resolve()}")

## 16. Inference Examples

In [ ]:
def load_artifacts_for_inference(model_dir):
    tokenizer = AutoTokenizer.from_pretrained(model_dir)
    model = AutoModelForSequenceClassification.from_pretrained(model_dir)
    with open(Path(model_dir) / "label_mapping.json", "r", encoding="utf-8") as f:
        mapping = json.load(f)
    return tokenizer, model, mapping

In [ ]:
def predict_text(text, model, tokenizer, mapping, max_length=256, threshold=0.5, device=None):
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"

    model.to(device)
    model.eval()

    cleaned_text = normalize_arabic_text(
        text,
        remove_urls=CFG.remove_urls,
        normalize_whitespace=CFG.normalize_whitespace,
        remove_emojis=CFG.remove_emojis,
        remove_tatweel=CFG.remove_tatweel,
        normalize_arabic=CFG.normalize_arabic,
        normalize_alef=CFG.normalize_alef,
        normalize_yeh=CFG.normalize_yeh,
        normalize_teh_marbuta=CFG.normalize_teh_marbuta,
    )

    encoded = tokenizer(
        cleaned_text,
        truncation=True,
        padding=True,
        max_length=max_length,
        return_tensors="pt"
    )

    encoded = {k: v.to(device) for k, v in encoded.items()}

    with torch.no_grad():
        outputs = model(**encoded)
        logits = outputs.logits

    task_type_local = mapping["task_type"]
    id2label_local = {int(k): v for k, v in mapping["id2label"].items()}

    if task_type_local == "multilabel":
        probs = torch.sigmoid(logits).cpu().numpy()[0]
        pred_ids = np.where(probs >= threshold)[0].tolist()
        pred_labels = [id2label_local[i] for i in pred_ids]
        return {
            "text": text,
            "cleaned_text": cleaned_text,
            "predicted_labels": pred_labels,
            "probabilities": {id2label_local[i]: float(probs[i]) for i in range(len(probs))}
        }
    else:
        probs = torch.softmax(logits, dim=1).cpu().numpy()[0]
        pred_id = int(np.argmax(probs))
        pred_label = id2label_local[pred_id]
        return {
            "text": text,
            "cleaned_text": cleaned_text,
            "predicted_label": pred_label,
            "predicted_label_id": pred_id,
            "probabilities": {id2label_local[i]: float(probs[i]) for i in range(len(probs))}
        }

In [ ]:
inf_tokenizer, inf_model, inf_mapping = load_artifacts_for_inference(CFG.save_dir)

example_texts = [
    "هذا الخبر يتعلق بالشأن الاقتصادي والاستثمار في المنطقة.",
    "الخدمة سيئة جدا ولم يتم حل المشكلة حتى الآن."
]

for txt in example_texts:
    result = predict_text(
        txt,
        model=inf_model,
        tokenizer=inf_tokenizer,
        mapping=inf_mapping,
        max_length=inf_mapping.get("max_length", 256)
    )
    print(json.dumps(result, ensure_ascii=False, indent=2))
    print("=" * 80)

## 17. Best Practices

- Start with **AraBERT** for MSA / formal Arabic.
- Try **MARBERT** if the data is dialectal or social-media-heavy.
- Use **weighted F1** as your main model-selection metric on imbalanced datasets.
- Avoid aggressive Arabic cleaning before Transformer fine-tuning.
- Keep the test set untouched until final evaluation.
- Save label mappings with the model.
- Track experiments consistently.

## Common Pitfalls

- Over-cleaning Arabic text and removing useful morphology
- Using accuracy only on imbalanced data
- Applying normalization that changes label-relevant spelling cues
- Forgetting stratification for single-label classification
- Choosing a model unsuited to the Arabic variety:
  - MSA vs dialectal Arabic
- Ignoring threshold tuning in multi-label problems
- Using oversampling blindly and overfitting duplicated minority samples

## Requirements

You can install dependencies with:

```bash
pip install -r requirements.txt
```

Or manually:

```bash
pip install torch transformers datasets accelerate scikit-learn imbalanced-learn pandas numpy matplotlib openpyxl jupyter
```

## Future Improvements

- Add **Optuna** for more efficient tuning
- Add **iterative stratification** for multi-label splits
- Add **threshold optimization** for binary and multi-label tasks
- Add **MLflow / Weights & Biases** experiment tracking
- Export to **ONNX** or **TorchScript** for deployment
- Add **FastAPI** serving
- Compare multiple Arabic models on the same dataset:
  - AraBERT
  - MARBERT
  - CAMeLBERT
  - ArabicBERT
- Add dialect-aware error analysis